# Email-Based Job Approval - End-to-End Test

This notebook tests the full email approval flow:

1. DS submits a job to DO
2. `notify` service sends DO an email notification with job details
3. DO replies **"approve"** (or "deny reason") via email
4. `email_approve` service detects reply via Gmail Pub/Sub push notifications
5. Job gets approved and executed automatically
6. DS downloads the results

### Prerequisites
- GCP project with **Gmail API** and **Pub/Sub API** enabled
- OAuth credentials JSON (`test_app_credentials.json`)
- DS token (`token_ds.json`)

## Configuration

Fill in your email addresses below.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

DO_EMAIL = "koenlennartvanderveen@gmail.com"
DS_EMAIL = "koen@openmined.org"

# Credential paths (relative to notebooks/)
APP_CREDENTIALS = Path("../credentials/test_app_credentials.json")
TOKEN_DS = Path("../credentials/token_ds.json")

print(f"App credentials: {APP_CREDENTIALS} - {'exists' if APP_CREDENTIALS.exists() else 'MISSING'}")
print(f"DS token: {TOKEN_DS} - {'exists' if TOKEN_DS.exists() else 'MISSING'}")

## Step 1: Create DO Drive token from app credentials

This converts the OAuth app credentials to a user-authorized Drive token.
It will open a browser window for you to authorize.

In [ ]:
import syft_client as sc
do_token_path = str(APP_CREDENTIALS.parent / "token_do_email_test.json")

# do_token_path = sc.credentials_to_token(
#     credentials_path=str(APP_CREDENTIALS),
#     output_path=do_token_path
# )
# print(f"DO token saved to: {do_token_path}")

## Step 2: Login as Data Owner and Data Scientist

In [ ]:
do_client = sc.login_do(email=DO_EMAIL, token_path=str(do_token_path))

In [ ]:
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

## Step 2a: possibly delete syftboxes

In [ ]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()

## Step 3: Set up peer relationship

DS requests access to DO, then DO approves.

In [ ]:
if False:
    ds_client.add_peer(DO_EMAIL)
    
    do_client.load_peers()
    do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)
    
    print("Peer relationship established")

## Step 4: Set up syft-bg

In [ ]:
import syft_bg

In [ ]:
cfg = syft_bg.config

In [ ]:
cfg

In [ ]:
!ls ~/.syft-bg/

In [ ]:
syft_bg.configure(do_email=DO_EMAIL, token_path=do_token_path)

In [ ]:
syft_bg.ensure_running(services=["email_approve"])

In [ ]:
syft_bg.status

In [ ]:
syft_bg.reset()

In [ ]:
import syft_bg

auth_result = syft_bg.authenticate(credentials_path=str(APP_CREDENTIALS))
auth_result

## Step 5: Initialize and start syft-bg services

We start:
- **notify**: sends email to DO when a new job arrives
- **email_approve**: monitors DO's Gmail for "approve"/"deny" replies via Pub/Sub push notifications

We disable auto-approval (`approve_jobs=False`) so jobs are only approved via email reply.

In [ ]:
import syft_bg

In [ ]:
# CREDS_DIR = Path.home() / ".syft-creds"

# result = syft_bg.init(
#     email=DO_EMAIL,
#     start=True,
#     skip_oauth=True,
#     credentials_path=str(APP_CREDENTIALS),
#     gmail_token_path=str(CREDS_DIR / "gmail_token.json"),
#     drive_token_path=str(do_token_path),
#     notify_jobs=True,
#     notify_peers=False,
#     approve_jobs=False,
#     approve_peers=False,
# )
# print(result)

In [ ]:
syft_bg.status

In [ ]:
!rm /Users/koen/.syft-creds/email_approve/daemon.log

In [ ]:
syft_bg.stop(service="email_approve")

In [ ]:
syft_bg.start(service="email_approve")

In [ ]:
syft_bg.logs(service="email_approve")

In [ ]:
!rm /Users/koen/.syft-creds/notify/daemon.log

In [ ]:
syft_bg.stop(service="notify")

In [ ]:
syft_bg.start(service="notify")

In [ ]:
syft_bg.logs(service="notify")

In [ ]:
# # Start the email_approve service (monitors Gmail replies via Pub/Sub)
# email_approve_result = syft_bg.start("email_approve")
# print("email_approve:", email_approve_result)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# syft_bg.logs(service="email_approve")

In [ ]:
# syft_bg.status

## Step 6: Create a test script

In [ ]:
import tempfile

SCRIPT_DIR = Path(tempfile.mkdtemp(prefix="email_approve_test_"))
script_path = SCRIPT_DIR / "main.py"

script_content = '''import os
import json

result = {"message": "Hello from email-approved job!", "value": 42}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
'''

script_path.write_text(script_content)
print(f"Script: {script_path}")
print(script_content)

## Step 7: DS submits the job

After submission, the `notify` service will detect the job and send an email to the DO.

In [ ]:
import uuid

JOB_NAME = f"email_test_{uuid.uuid4().hex[:8]}"

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(script_path),
    job_name=JOB_NAME,
    force_submission=True,
)
print(f"Job '{JOB_NAME}' submitted to {DO_EMAIL}")

In [ ]:
# ds_client.jobs

In [ ]:
# DO syncs to receive the job
do_client.sync()
print(f"DO jobs: {do_client.job_client.jobs}")

## Step 8: Check your email and reply "approve"

1. Open the **DO's Gmail inbox**
2. Find the job notification email (subject like "New Job: ...")
3. Reply with just the word: **`approve`**
4. The `email_approve` service will detect the reply and execute the job automatically

To reject instead, reply: `deny <reason>`

In [ ]:
import time

print("Waiting for email approval...")
print(f"Check {DO_EMAIL}'s inbox and reply 'approve' to the job notification.")
print()

start_time = time.time()
TIMEOUT = 300  # 5 minutes

while time.time() - start_time < TIMEOUT:
    do_client.sync()
    matching = [j for j in do_client.job_client.jobs if j.name == JOB_NAME]
    if matching:
        job = matching[0]
        elapsed = time.time() - start_time
        print(f"  [{elapsed:.0f}s] Job status: {job.status}")
        if job.status == "done":
            print(f"\nJob completed!")
            break
        elif job.status == "failed":
            print(f"\nJob failed!")
            break
    else:
        print(f"  [{time.time() - start_time:.0f}s] Job not yet visible")
    time.sleep(10)
else:
    print(f"\nTimeout after {TIMEOUT}s. Check email_approve logs below.")

## Step 9: DS downloads results

In [ ]:
ds_client.sync()

ds_jobs = [j for j in ds_client.job_client.jobs if j.name == JOB_NAME]
if ds_jobs:
    ds_job = ds_jobs[0]
    print(f"Job: {ds_job.name}")
    print(f"Status: {ds_job.status}")
    print(f"Output paths: {ds_job.output_paths}")
    print()
    for p in ds_job.output_paths:
        print(f"=== {p.name} ===")
        print(p.read_text())
else:
    print("Job not found yet - re-run this cell after a few seconds")

## Debugging: Service logs

In [ ]:
print("=== email_approve ===")
for line in syft_bg.logs("email_approve", n=30):
    print(line)

print("\n=== notify ===")
for line in syft_bg.logs("notify", n=20):
    print(line)

## Cleanup

In [ ]:
syft_bg.stop()


In [ ]:
syft_bg.status?